In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
!pip -q install torch pandas numpy scikit-learn tqdm

In [3]:
import re
import random
import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

In [4]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)

Device: cpu


In [5]:
train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

test = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv")

sample_submission = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv")

print(train.shape)

print(test.shape)

train.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [6]:
OPTION_LETTERS = ["A","B","C","D","E"]

label_map = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4,
}

inverse_label_map = {
    v:k
    for k,v in label_map.items()
}


def build_text(row):

    return (
        f"Question: {row['prompt']} "
        f"A: {row['A']} "
        f"B: {row['B']} "
        f"C: {row['C']} "
        f"D: {row['D']} "
        f"E: {row['E']}"
    )


train["text"] = train.apply(
    build_text,
    axis=1,
)

test["text"] = test.apply(
    build_text,
    axis=1,
)

train["label"] = train["answer"].map(
    label_map
)

train[["text","answer","label"]].head()

,text,answer,label
0,Question: Pick the best possible answer: What ...,B,1
1,Question: What is accelerator-based light-ion ...,A,0
2,Question: Determine the correct option: What i...,C,2
3,Question: Select the most accurate option: Wha...,B,1
4,Question: Identify the correct statement: What...,A,0


In [7]:
def tokenize(text):

    text = text.lower()

    text = re.sub(r"[^a-z0-9\s]", " ", text)

    tokens = text.split()

    return tokens



print(tokenize(train["text"].iloc[0])[:25])

['question', 'pick', 'the', 'best', 'possible', 'answer', 'what', 'is', 'martin', 'heidegger', 's', 'view', 'on', 'the', 'relationship', 'between', 'time', 'and', 'human', 'existence', 'among', 'the', 'listed', 'options', 'a']


In [8]:
SPECIAL_TOKENS = {

    "<PAD>":0,

    "<UNK>":1,

}

word2idx = dict(SPECIAL_TOKENS)

idx2word = {

    0:"<PAD>",

    1:"<UNK>",

}

word_freq = {}

for text in train["text"]:

    for token in tokenize(text):

        word_freq[token] = word_freq.get(token,0) + 1


MIN_FREQ = 2

for word,count in sorted(word_freq.items()):

    if count >= MIN_FREQ:

        idx = len(word2idx)

        word2idx[word] = idx

        idx2word[idx] = word

print("Vocabulary Size:",len(word2idx))

Vocabulary Size: 2973


In [9]:
MAX_LEN = 384


def encode(text):

    tokens = tokenize(text)

    ids = []

    for token in tokens:

        ids.append(

            word2idx.get(

                token,

                word2idx["<UNK>"]

            )

        )

    

    ids = ids[:MAX_LEN]

    

    if len(ids) < MAX_LEN:

        ids += [

            word2idx["<PAD>"]

        ] * (MAX_LEN-len(ids))

    return ids

In [10]:
sample = encode(train["text"].iloc[0])

print(sample[:20])

print(len(sample))

[2174, 2007, 2662, 324, 2056, 186, 2925, 1450, 1636, 1251, 2347, 2881, 1875, 2662, 2264, 326, 2700, 180, 1297, 985]
384


In [11]:
train_df, val_df = train_test_split(

    train,

    test_size=0.20,

    random_state=SEED,

    stratify=train["label"],

)

print(train_df.shape)

print(val_df.shape)

(1600, 10)
(400, 10)


In [12]:
class MCQDataset(Dataset):

    def __init__(self, dataframe, train_mode=True):

        self.df = dataframe.reset_index(drop=True)

        self.train_mode = train_mode


    def __len__(self):

        return len(self.df)


    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        x = torch.tensor(

            encode(row["text"]),

            dtype=torch.long,

        )

        if self.train_mode:

            y = torch.tensor(

                row["label"],

                dtype=torch.long,

            )

            return x,y

        return x


train_dataset = MCQDataset(train_df)

val_dataset = MCQDataset(val_df)

test_dataset = MCQDataset(

    test,

    train_mode=False,

)

train_loader = DataLoader(

    train_dataset,

    batch_size=32,

    shuffle=True,

)

val_loader = DataLoader(

    val_dataset,

    batch_size=64,

)

test_loader = DataLoader(

    test_dataset,

    batch_size=64,

)

print("Train batches:",len(train_loader))

print("Validation batches:",len(val_loader))

Train batches: 50
Validation batches: 7


In [13]:
x,y = next(iter(train_loader))

print(x.shape)

print(y.shape)

torch.Size([32, 384])
torch.Size([32])


In [14]:
class BiLSTMClassifier(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=200,
        hidden_dim=256,
        num_layers=1,
        num_classes=5,
        dropout=0.4,
    ):

        super().__init__()

        
        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=0,
        )

        
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
        )

        self.dropout = nn.Dropout(dropout)

        
        self.fc = nn.Linear(
            hidden_dim * 2,
            num_classes,
        )

    def forward(self, x):

        x = self.embedding(x)

        outputs, (hidden, cell) = self.lstm(x)

        
        forward_hidden = hidden[-2]

        
        backward_hidden = hidden[-1]

        hidden = torch.cat(
            (forward_hidden, backward_hidden),
            dim=1,
        )

        hidden = self.dropout(hidden)

        logits = self.fc(hidden)

        return logits

In [15]:
model = BiLSTMClassifier(

    vocab_size=len(word2idx),

    embedding_dim=200,

    hidden_dim=256,

    num_layers=1,

    num_classes=5,

    dropout=0.4,

).to(DEVICE)

print(model)

BiLSTMClassifier(
  (embedding): Embedding(2973, 200, padding_idx=0)
  (lstm): LSTM(200, 256, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.4, inplace=False)
  (fc): Linear(in_features=512, out_features=5, bias=True)
)


In [16]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(

    model.parameters(),

    lr=1e-3,

)

scheduler = torch.optim.lr_scheduler.StepLR(

    optimizer,

    step_size=3,

    gamma=0.5,

)

In [17]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(f"Total Parameters : {total_params:,}")

print(f"Trainable Parameters : {trainable_params:,}")

Total Parameters : 1,535,149
Trainable Parameters : 1,535,149


In [18]:
def train_one_epoch(model, loader, optimizer, criterion):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in tqdm(loader):

        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()

        total += labels.size(0)

    epoch_loss = running_loss / len(loader)
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [19]:
@torch.no_grad()
def evaluate(model, loader, criterion):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in loader:

        inputs = inputs.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(inputs)

        loss = criterion(outputs, labels)

        running_loss += loss.item()

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()

        total += labels.size(0)

    val_loss = running_loss / len(loader)
    val_acc = correct / total

    return val_loss, val_acc

In [20]:
EPOCHS = 10

best_val_acc = 0.0

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
}

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
    )

    val_loss, val_acc = evaluate(
        model,
        val_loader,
        criterion,
    )

    scheduler.step()

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(f"Train Loss : {train_loss:.4f}")
    print(f"Train Acc  : {train_acc:.4f}")
    print(f"Val Loss   : {val_loss:.4f}")
    print(f"Val Acc    : {val_acc:.4f}")

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(
            model.state_dict(),
            "best_bilstm_model.pth",
        )

        print(" Best model saved")


Epoch 1/10


  0%|          | 0/50 [00:00<?, ?it/s]

Train Loss : 1.6045
Train Acc  : 0.2487
Val Loss   : 1.5745
Val Acc    : 0.2450
 Best model saved

Epoch 2/10


  0%|          | 0/50 [00:00<?, ?it/s]

Train Loss : 1.5846
Train Acc  : 0.2394
Val Loss   : 1.5706
Val Acc    : 0.2525
 Best model saved

Epoch 3/10


...

Train Loss : 0.0241
Train Acc  : 0.9975
Val Loss   : 0.0563
Val Acc    : 0.9875
 Best model saved

Epoch 9/10


  0%|          | 0/50 [00:00<?, ?it/s]

Train Loss : 0.0195
Train Acc  : 0.9975
Val Loss   : 0.0535
Val Acc    : 0.9875

Epoch 10/10


  0%|          | 0/50 [00:00<?, ?it/s]

Train Loss : 0.0180
Train Acc  : 0.9975
Val Loss   : 0.0478
Val Acc    : 0.9875


In [21]:
model.load_state_dict(

    torch.load(
        "best_bilstm_model.pth",
        map_location=DEVICE,
    )

)

model.eval()

print("Best model loaded.")

Best model loaded.


In [22]:
import pandas as pd

history_df = pd.DataFrame(history)

history_df

,train_loss,train_acc,val_loss,val_acc
0,1.604471,0.248750,1.574458,0.2450
1,1.584572,0.239375,1.570603,0.2525
2,1.466387,0.349375,1.072236,0.5750
3,0.623061,0.813125,0.286202,0.9350
4,0.164052,0.960625,0.105315,0.9675
5,0.051740,0.992500,0.121368,0.9650
6,0.036119,0.996250,0.071445,0.9825
7,0.024096,0.997500,0.056275,0.9875
8,0.019496,0.997500,0.053517,0.9875
9,0.018008,0.997500,0.047840,0.9875


In [23]:
@torch.no_grad()
def predict(loader):

    model.eval()

    predictions = []

    probabilities = []

    for inputs in tqdm(loader):

        inputs = inputs.to(DEVICE)

        outputs = model(inputs)

        probs = torch.softmax(outputs, dim=1)

        probabilities.extend(
            probs.cpu().numpy()
        )

        top3 = torch.argsort(
            probs,
            dim=1,
            descending=True,
        )[:, :3]

        predictions.extend(
            top3.cpu().numpy()
        )

    return predictions, probabilities

In [24]:
predictions, probabilities = predict(
    test_loader
)

print(len(predictions))

  0%|          | 0/8 [00:00<?, ?it/s]

500


In [25]:
submission_predictions = []

for pred in predictions:

    letters = [

        inverse_label_map[idx]

        for idx in pred

    ]

    submission_predictions.append(

        " ".join(letters)

    )

submission_predictions[:10]

['A C B',
 'B C A',
 'B C E',
 'E B D',
 'C B A',
 'D E B',
 'E B D',
 'B C E',
 'C B A',
 'B C E']

In [26]:
submission = pd.DataFrame({

    "ID": test["id"],

    "Prediction": submission_predictions,

})

submission.head()

,ID,Prediction
0,1,A C B
1,2,B C A
2,3,B C E
3,4,E B D
4,5,C B A


In [27]:
submission.to_csv(

    "submission.csv",

    index=False,

)

print("submission.csv saved!")

submission.head()

submission.csv saved!


,ID,Prediction
0,1,A C B
1,2,B C A
2,3,B C E
3,4,E B D
4,5,C B A


In [28]:
import wandb

run = wandb.init(

    entity="mrinal-pandey2905-pes-university",

    project="23f2000333-t22026",

    name="BiLSTM",

    job_type="training",

    config={

        "architecture":"BiLSTM",

        "embedding_dim":200,

        "hidden_dim":256,

        "layers":1,

        "dropout":0.4,

        "epochs":EPOCHS,

        "optimizer":"Adam",

        "learning_rate":1e-3,

        "batch_size":32,

        "max_length":MAX_LEN,

        "loss":"CrossEntropyLoss",

        "dataset":"Smart MCQ Solver Challenge",

    }

)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [29]:
best_epoch = int(np.argmax(history["val_acc"]))

wandb.log({

    "best_validation_accuracy": max(history["val_acc"]),

    "final_training_accuracy": history["train_acc"][-1],

    "final_validation_accuracy": history["val_acc"][-1],

    "best_validation_loss": min(history["val_loss"]),

    "best_epoch": best_epoch + 1,

})

In [30]:
artifact = wandb.Artifact(

    "bilstm-model",

    type="model",

)

artifact.add_file(

    "best_bilstm_model.pth"

)

run.log_artifact(artifact)

run.finish()

best_epoch,▁
best_validation_accuracy,▁
best_validation_loss,▁
final_training_accuracy,▁
final_validation_accuracy,▁
best_epoch,8
best_validation_accuracy,0.9875
best_validation_loss,0.04784
final_training_accuracy,0.9975
final_validation_accuracy,0.9875
